# Customer Persona Discovery via PCA & Clustering
**Author:** Product / Business Analyst Candidate
**Dataset:** Synthetic Indian E-Commerce Customer Behavioral Dataset (18,497 Delivered Customers)

## 1. Imports & Data Loading

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

%matplotlib inline
plt.rcParams['figure.figsize'] = (10, 6)


## 2. Load Features & Diagnostics

In [ ]:
df_feat = pd.read_csv('../outputs/customer_features.csv')
print('Feature table shape:', df_feat.shape)
df_feat.head()


## 3. Before-PCA Diagnostic: Feature Correlation Heatmap

In [ ]:
df_model = df_feat.set_index('customer_id')
model_cols = [
    'recency_days', 'tenure_days', 'order_frequency', 'is_repeat_customer',
    'log_monetary_value', 'log_avg_order_value', 'avg_basket_size',
    'total_items_purchased', 'category_diversity', 'cat_apparel_ratio',
    'cat_electronics_ratio', 'cat_groceries_ratio', 'discount_order_pct',
    'avg_discount_pct', 'return_rate', 'cancellation_rate', 'avg_review_score',
    'review_response_rate', 'weekend_order_ratio', 'log_days_between_orders',
    'cod_payment_ratio', 'upi_payment_ratio', 'avg_delivery_delay_days'
]
X = df_model[model_cols]
plt.figure(figsize=(14, 11))
sns.heatmap(X.corr(), annot=True, fmt='.2f', cmap='coolwarm')
plt.title('Before-PCA Correlation Heatmap')
plt.show()


## 4. Preprocessing & StandardScaler

In [ ]:
scaler = StandardScaler()
X_scaled = pd.DataFrame(scaler.fit_transform(X), index=X.index, columns=X.columns)
print('Scaled features shape:', X_scaled.shape)


## 5. Principal Component Analysis (PCA) & Scree Plot

In [ ]:
pca = PCA()
X_pca = pd.DataFrame(pca.fit_transform(X_scaled), index=X.index, columns=[f'PC{i+1}' for i in range(X.shape[1])])
cum_var = np.cumsum(pca.explained_variance_ratio_)

fig, ax1 = plt.subplots(figsize=(10, 5))
ax1.bar(range(1, 24), pca.explained_variance_ratio_, alpha=0.6, color='b', label='Individual Variance')
ax2 = ax1.twinx()
ax2.plot(range(1, 24), cum_var, color='r', marker='o', label='Cumulative Variance')
ax2.axhline(0.80, color='g', linestyle='--', label='80% Threshold')
plt.title('PCA Scree Plot')
plt.show()

print(f'11 Components Variance: {cum_var[10]*100:.2f}%')


## 6. Clustering Evaluation: 11 PCs vs Raw Features

In [ ]:
X_pca_11 = X_pca[[f'PC{i+1}' for i in range(11)]]
for k in range(2, 7):
    km_r = KMeans(n_clusters=k, random_state=42, n_init=10).fit(X_scaled)
    sil_r = silhouette_score(X_scaled, km_r.labels_, sample_size=3000)
    km_p = KMeans(n_clusters=k, random_state=42, n_init=10).fit(X_pca_11)
    sil_p = silhouette_score(X_pca_11, km_p.labels_, sample_size=3000)
    print(f'k={k} | Raw Sil: {sil_r:.4f} | PCA-11 Sil: {sil_p:.4f} | Diff: {((sil_p-sil_r)/sil_r)*100:+.2f}%')


## 7. Persona Visualization & Summary Table

In [ ]:
clustered = pd.read_csv('../outputs/clustered_customer_features.csv')
summary = pd.read_csv('../outputs/persona_summary_table.csv')
summary
